# 01 · Datos y parches — Dani
Sólo bloque 1: entorno, camera/coins, extracción, muestreo y ensamblado.
El módulo `.py` es la fuente de verdad. Ejecutar todas las celdas en orden, con CPU.
No se entrena K-SVD ni se calculan métricas de denoising.


In [ ]:
from pathlib import Path
import sys, importlib, json, shutil, hashlib
import numpy as np
# Comprobar dependencias existentes; no actualizar el entorno automáticamente.
for package in ('numpy', 'scipy', 'sklearn', 'skimage', 'matplotlib'):
    module = importlib.import_module(package)
    print(package, module.__version__)
KAGGLE = Path('/kaggle/working').is_dir()
# Si hay varias copias, indicar explícitamente la ruta real en SOURCE_DIR.
SOURCE_DIR = None
if SOURCE_DIR is None:
    if KAGGLE:
        candidates = list(Path('/kaggle/input').rglob('preprocesamiento.py'))
    else:
        candidates = [p / 'src/preprocesamiento.py' for p in (Path.cwd(), *Path.cwd().parents)
                      if (p / 'src/preprocesamiento.py').is_file()]
        if not candidates and (Path.cwd() / 'preprocesamiento.py').is_file():
            candidates = [Path.cwd() / 'preprocesamiento.py']
    if len(candidates) != 1:
        raise RuntimeError(f'Indica SOURCE_DIR usando la ruta real. Candidatos: {candidates}')
    SOURCE_DIR = candidates[0].parent
SOURCE_DIR = Path(SOURCE_DIR).resolve()
sys.path.insert(0, str(SOURCE_DIR))
from preprocesamiento import save_data, extract_patches, assemble_patches, validate_data
OUTPUT = Path('/kaggle/working/entrega_01') if KAGGLE else (SOURCE_DIR.parent / 'artifacts/entrega_01' if SOURCE_DIR.name == 'src' else Path.cwd() / 'artifacts/entrega_01')
print('Fuente:', SOURCE_DIR, '\nSalida:', OUTPUT)


## Preparación y muestreo
`save_data` convierte con `img_as_float64` antes de redimensionar. Extrae columnas
centradas y escoge 1500 parches no constantes con RNG 42. Los índices apuntan a
la extracción completa. Conserva los parámetros de modelo y ruido sin ejecutarlos.


In [ ]:
evidence = save_data(OUTPUT)
print(json.dumps(evidence, indent=2))
with np.load(OUTPUT / 'datos.npz', allow_pickle=False) as loaded:
    arrays = dict(loaded)
validate_data(arrays)
for key, value in arrays.items():
    print(key, value.shape, value.dtype)


## Control de inversión y bordes
Cada columna tiene 64 valores. Para reconstruir hay que sumar su media una sola
vez. El ensamblado promedia las contribuciones, también en esquinas y bordes.
Los siguientes casos son controles sintéticos, no resultados experimentales.


In [ ]:
for image in (np.full((13, 15), 0.3), np.arange(11*17, dtype=np.float64).reshape(11, 17)/10-2):
    Z, means, positions = extract_patches(image)
    restored = assemble_patches(Z + means[None, :], positions, image.shape)
    error = float(np.max(np.abs(restored-image)))
    assert error < 1e-10
    assert len(np.unique(positions, axis=0)) == len(positions)
    print('Control', image.shape, 'error máximo:', error)
print('Controles de imagen constante y tamaño irregular aprobados.')


## Inspección visual
Las figuras muestran las imágenes separadas y los primeros 16 parches de la
muestra, con su media restituida para poder verlos. No son átomos aprendidos.


In [ ]:
from IPython.display import display, Image, Markdown
display(Image(filename=str(OUTPUT / 'imagenes.png')))
display(Image(filename=str(OUTPUT / 'parches.png')))
display(Markdown((OUTPUT / 'nota_datos.md').read_text(encoding='utf-8')))


## Exportación
La copia del módulo es idéntica. Kaggle conserva el notebook ejecutado al guardar
una versión con salidas; el archivo copiado abajo es su fuente. Descargar además
el ZIP de respaldo. El manifiesto permite detectar cambios durante el traslado.
En local, `scripts/ejecutar_notebook.py` añade al paquete el notebook ejecutado.


In [ ]:
shutil.copyfile(SOURCE_DIR / 'preprocesamiento.py', OUTPUT / 'preprocesamiento.py')
assert (OUTPUT / 'preprocesamiento.py').read_bytes() == (SOURCE_DIR / 'preprocesamiento.py').read_bytes()
notebook_candidates = [SOURCE_DIR / '01_datos.ipynb', SOURCE_DIR.parent / 'notebooks/01_datos.ipynb']
notebook_source = next((p for p in notebook_candidates if p.is_file()), None)
if notebook_source is None:
    raise FileNotFoundError('Adjunta también 01_datos.ipynb en el recurso de entrada.')
shutil.copyfile(notebook_source, OUTPUT / '01_datos.ipynb')
manifest = {p.name: hashlib.sha256(p.read_bytes()).hexdigest()
            for p in OUTPUT.iterdir() if p.is_file() and p.name != 'manifest.sha256.json'}
(OUTPUT / 'manifest.sha256.json').write_text(json.dumps(manifest, indent=2), encoding='utf-8')
archive = shutil.make_archive(str(OUTPUT.parent / 'entrega_01_v1'), 'zip', OUTPUT)
print('Respaldo:', archive)
print('Verificado en:', evidence['execution_environment'])


## Siguiente integrante
Cargar `datos.npz` con `allow_pickle=False`, verificar el manifiesto y leer
`config.json`. Usar `X_train` (64,1500) para el modelo; reservar `test_image`.
Fundamento: [Elad y Aharon, 2006](https://doi.org/10.1109/TIP.2006.881969).
Documentación: [skimage.data](https://scikit-image.org/docs/stable/api/skimage.data.html).
